In [ ]:
from pathlib import Path
import geopandas as gpd

data_path = Path("../data/raw/vectors/raw_data/geometries/Estonia.geojson")

estonia_gdf = gpd.read_file(data_path)

estonia_gdf.head()

,parcel_id,geometry
0,19994165,"POLYGON ((26.50243 59.31839, 26.50244 59.31843..."
1,19990783,"POLYGON ((24.54648 58.86884, 24.54674 58.86879..."
2,19990784,"POLYGON ((24.54597 58.86827, 24.54668 58.86816..."
3,19996106,"POLYGON ((27.42837 58.11975, 27.42839 58.11972..."
4,19990620,"POLYGON ((26.66816 57.82049, 26.66815 57.8205,..."


In [ ]:
estonia_wgs84 = estonia_gdf.to_crs("EPSG:4326")

field = estonia_wgs84.iloc[0]

minx, miny, maxx, maxy = field.geometry.bounds

bbox = [minx, miny, maxx, maxy]

print(bbox)

[26.50226345, 59.31772336, 26.50391273, 59.31879163]


### Getting the raster file from the STAC API

In [ ]:
import leafmap.foliumap as leafmap

### Helper code from Hendrik to make our own subset tif
I want a small study area to see the raster and the vector data together, so I concentrate on a field location in Estonia. 
The code below creates some functions that I can use to create such a file. Then I will use leaflet to visualize. 

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from pyproj import Transformer
from rasterio.windows import from_bounds
from rasterio.windows import transform as window_transform
 
def bbox_to_dataset_crs(bbox_wgs84, dst_crs):
    """Transform a WGS84 bounding box to a raster's native CRS.

    Rasterio windows must be given in the source file's own CRS, not WGS84.

    Args:
        bbox_wgs84: Bounding box in WGS84 [min_lon, min_lat, max_lon, max_lat].
        dst_crs: Target CRS (e.g., src.crs from rasterio.open()).

    Returns:
        Bounding box in the target CRS [min_x, min_y, max_x, max_y].
    """
    transformer = Transformer.from_crs("EPSG:4326", dst_crs, always_xy=True)
    min_lon, min_lat, max_lon, max_lat = bbox_wgs84
    xs, ys = transformer.transform([min_lon, max_lon], [min_lat, max_lat])
    return min(xs), min(ys), max(xs), max(ys)


def ycbcr_to_rgb(stack):
    """Convert a 3-band YCbCr image (ITU-R BT.601) to RGB uint8.

    Some imagery (e.g. JPEG-compressed GeoTIFFs, as used by SWISSIMAGE) stores
    colour as YCbCr rather than RGB, and must be converted before it can be
    written or plotted as a normal RGB image.

    Args:
        stack: Array of shape (3, height, width) with YCbCr bands.

    Returns:
        RGB array of shape (3, height, width) as uint8.
    """
    y, cb, cr = stack.astype("float32")
    r = y + 1.402 * (cr - 128.0)
    g = y - 0.344136 * (cb - 128.0) - 0.714136 * (cr - 128.0)
    b = y + 1.772 * (cb - 128.0)
    return np.clip(np.stack([r, g, b]), 0, 255).astype("uint8")


def save_single_asset_subset(
    item, asset_key, bbox_wgs84, output_path, band_indexes=(1, 2, 3)
):
    """Read the AOI window from one raster asset and save it as a GeoTIFF.

    Output pixel dimensions follow the AOI at the source's native resolution,
    so a landscape bbox produces a landscape image (no separate width/height
    arguments needed).

    Args:
        item: A STAC Item object.
        asset_key: String key of the asset to read (e.g., 'B02', 'red', 'image').
        bbox_wgs84: Bounding box in WGS84 [min_lon, min_lat, max_lon, max_lat].
        output_path: File path for the output GeoTIFF.
        band_indexes: Tuple of band indices to read (1-indexed; default (1,2,3) for RGB).

    Returns:
        Path to the written GeoTIFF file.
    """
    href = item.assets[asset_key].href

    with rasterio.open(href) as src:
        print("Source CRS:", src.crs)
        bounds = bbox_to_dataset_crs(bbox_wgs84, src.crs)
        window = from_bounds(*bounds, transform=src.transform)

        stack = src.read(
            list(band_indexes), window=window, boundless=True, fill_value=0
        )
        print("Read shape, dtype:", stack.shape, stack.dtype)

        profile = src.profile.copy()
        profile.update(
            driver="GTiff",
            height=stack.shape[1],
            width=stack.shape[2],
            count=len(band_indexes),
            transform=window_transform(window, src.transform),
            compress="deflate",
        )
        profile.pop("photometric", None)

        if (
            str(src.profile.get("photometric", "")).lower() == "ycbcr"
            and stack.shape[0] == 3
        ):
            print("Converting YCbCr -> RGB")
            stack = ycbcr_to_rgb(stack)
        profile.update(dtype=str(stack.dtype))

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(stack)

    size_mb = output_path.stat().st_size / 1_000_000
    print(f"Saved: {output_path} ({size_mb:.2f} MB)")
    return output_path


def save_multiband_subset(item, asset_keys, bbox_wgs84, output_path):
    """Read the AOI window from several single-band assets and save as one GeoTIFF.

    Each asset key is read separately (one band each) and stacked in the given
    order, e.g. asset_keys=['B04', 'B03', 'B02'] for a red-green-blue stack.
    Output pixel dimensions follow the AOI at the source's native resolution.

    Args:
        item: A STAC Item object.
        asset_keys: Ordered list of asset keys, one per output band.
        bbox_wgs84: Bounding box in WGS84 [min_lon, min_lat, max_lon, max_lat].
        output_path: File path for the output GeoTIFF.

    Returns:
        Path to the written GeoTIFF file.
    """
    bands = []
    profile = None

    for key in asset_keys:
        with rasterio.open(item.assets[key].href) as src:
            bounds = bbox_to_dataset_crs(bbox_wgs84, src.crs)
            window = from_bounds(*bounds, transform=src.transform)
            band = src.read(1, window=window, boundless=True, fill_value=0)
            bands.append(band)

            if profile is None:
                profile = src.profile.copy()
                profile.update(
                    driver="GTiff",
                    height=band.shape[0],
                    width=band.shape[1],
                    count=len(asset_keys),
                    transform=window_transform(window, src.transform),
                    compress="deflate",
                )
                profile.pop("photometric", None)

    stack = np.stack(bands)
    profile.update(dtype=str(stack.dtype))

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(stack)

    size_mb = output_path.stat().st_size / 1_000_000
    print(f"Saved: {output_path} ({size_mb:.2f} MB)")
    return output_path


def stretch_rgb(rgb):
    """Apply a percentile stretch to an RGB image for better visual contrast.

    Raw satellite data often spans a wide dynamic range. A 2-98 percentile stretch clips
    extreme values and maps the middle 96% to the display range, improving visibility of
    detail without losing important information to saturation.

    Args:
        rgb: Array of shape (3, height, width) or (height, width, 3).

    Returns:
        Stretched array normalized to [0, 1].
    """
    rgb = np.moveaxis(rgb, 0, -1).astype("float32")
    p_low, p_high = np.nanpercentile(rgb, (2, 98))
    if p_high == p_low:
        return np.zeros_like(rgb)
    rgb = (rgb - p_low) / (p_high - p_low)
    return np.clip(rgb, 0, 1)


def plot_rgb(path, title):
    """Load a GeoTIFF and display it as an RGB image with percentile stretch.

    Args:
        path: File path to a 3-band GeoTIFF.
        title: Title for the plot.
    """
    with rasterio.open(path) as src:
        rgb = src.read([1, 2, 3])
    plt.figure(figsize=(7, 3))
    plt.imshow(stretch_rgb(rgb))
    plt.title(title)
    plt.axis("off")
    plt.show()

### First lets get our scene in Estonia from the STAC API. 

In [ ]:
from pystac_client import Client
import planetary_computer as pc

pc_catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")

estonia_bbox = [25.609131,58.940742,27.553711,59.514414]  # WGS84 lon/lat somewhere in Estonia

search = pc_catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=estonia_bbox,
    datetime="2021-01-01/2021-12-31",
    query={"eo:cloud_cover": {"lt": 20}},
    max_items=10,
)

items = list(search.items())
print(f"Found {len(items)} Sentinel-2 item(s)")

Found 10 Sentinel-2 item(s)


Now we have 10 scenes from Sentinel 2, we want to choose the least cloudy scene. 

In [ ]:
items_sorted = sorted(
    items,
    key=lambda item: item.properties.get("eo:cloud_cover", 999),
)

s2_item = pc.sign(items_sorted[0]) #this is now signed to STAC so I can access it 

print("Selected item:", s2_item.id)
print("Date:", s2_item.datetime)
print("Cloud cover:", round(s2_item.properties.get("eo:cloud_cover"), 2))

print("\nUseful assets:")
for key in ["B02", "B03", "B04", "B08", "SCL", "visual"]:
    if key in s2_item.assets:
        print(key, "-", s2_item.assets[key].title)

Selected item: S2B_MSIL2A_20211102T093049_R136_T35VMF_20230128T232313
Date: 2021-11-02 09:30:49.024000+00:00
Cloud cover: 0.68

Useful assets:
B02 - Band 2 - Blue - 10m
B03 - Band 3 - Green - 10m
B04 - Band 4 - Red - 10m
B08 - Band 8 - NIR - 10m
SCL - Scene classfication map (SCL)
visual - True color image


We now have our least cloudy scene signed to the STAC API, this lets me access it through the API and create my own subset file.

In [ ]:
s2_path = save_multiband_subset(
    item=s2_item,
    asset_keys=[
        "B02", #Blue
        "B03", #Green
        "B04", #Red
        "B08", #NIR
    ],
    bbox_wgs84=estonia_bbox,
    output_path="../data/raw/imagery/sentinel2/estonia_field_0.tif",
)

Saved: ../data/raw/imagery/sentinel2/estonia_field_0.tif (263.60 MB)


### Now that we have it saved as tif, we can add it to leaflet for some visualization. 

In [ ]:
import leafmap

m3 = leafmap.Map(height="500px")

m3.add_basemap("Esri.WorldImagery") #basemap

m3.add_raster(
    str(s2_path), #s2path is where our GeoTIFF file is located. We read that and then add it to our basemap. 
    indexes=[3,2,1], #this is a regular RGB image. True color composite
    layer_name="Sentinel-2 True Color"
)
m3.add_raster(
    str(s2_path),
    indexes=[4,3,2], #this is a false color image using the NIR band for red and getting rid of blue.
    layer_name ="Sentinel-2 False Color"  
)
m3.add_layer_control()
m3

Map(center=[59.228159500000004, 26.5697985], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_…

### Lets add vector data to this map.

In [ ]:
print(estonia_gdf.geom_type.value_counts())
estonia_gdf.crs #check CRS
estonia_gdf.head()

Polygon         176054
MultiPolygon         1
Name: count, dtype: int64


,parcel_id,geometry
0,19994165,"POLYGON ((26.50243 59.31839, 26.50244 59.31843..."
1,19990783,"POLYGON ((24.54648 58.86884, 24.54674 58.86879..."
2,19990784,"POLYGON ((24.54597 58.86827, 24.54668 58.86816..."
3,19996106,"POLYGON ((27.42837 58.11975, 27.42839 58.11972..."
4,19990620,"POLYGON ((26.66816 57.82049, 26.66815 57.8205,..."


#### If we try and map all of these it will take forever. Best to just create a subset first. 


In [ ]:
estonia_subset = estonia_gdf.head(1)

In [ ]:
m3.add_gdf(estonia_subset, layer_name="Crops in part of Estonia", zoom_to_layer=True)
m3

Map(center=[59.228159500000004, 26.5697985], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_…

Current issues:

Raster data and the vector data are not in the same spot. 

How can we load more vector data, how much will I be able to load to visualize?

#### Solution, take a field and make it the center. Then get a bunch of other vector fields that are within 1km of that field.

Also make a bounding box for this field and get a small raster subset that we can sign to our STAC.

In [ ]:
# Project to a metric CRS
estonia_projected = estonia_gdf.to_crs("EPSG:32635")

center_field = estonia_projected.iloc[0]

distances = estonia_projected.geometry.centroid.distance(
    center_field.geometry.centroid
)

estonia_local = estonia_gdf.loc[distances[distances < 1000].index] 
estonia_local

,parcel_id,geometry
0,19994165,"POLYGON ((26.50243 59.31839, 26.50244 59.31843..."
320,19994164,"POLYGON ((26.50425 59.31763, 26.50429 59.31763..."
408,19994166,"POLYGON ((26.50002 59.31813, 26.50003 59.31813..."
494,19994167,"POLYGON ((26.49936 59.31789, 26.49948 59.31779..."
5335,20088271,"POLYGON ((26.50341 59.32013, 26.5034 59.31992,..."
5336,20088280,"POLYGON ((26.50687 59.31848, 26.50706 59.3184,..."
7709,20088270,"POLYGON ((26.50706 59.3233, 26.50775 59.32367,..."
29426,20384465,"POLYGON ((26.49671 59.31783, 26.49676 59.31781..."
30393,20384486,"POLYGON ((26.50052 59.32181, 26.50087 59.3218,..."
32260,20384502,"POLYGON ((26.49496 59.31897, 26.49496 59.31894..."


In [ ]:
##Getting raster subset from bounding box
#Making bounding box 
bbox_wgs84 = list(estonia_local.total_bounds)

print(bbox_wgs84)

#Getting the new raster
search = pc_catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox_wgs84,
    datetime="2021-05-01/2021-09-30",
    query={"eo:cloud_cover": {"lt": 20}},
    max_items=20,
)

items = list(search.items())
print(f"Found {len(items)} Sentinel-2 item(s)")

#Choosing least cloudy scene
items_sorted = sorted(
    items,
    key=lambda item: item.properties.get("eo:cloud_cover", 999),
)

s2_item = pc.sign(items_sorted[0]) #this is now signed to STAC so I can access it 

print("Selected item:", s2_item.id)
print("Date:", s2_item.datetime)
print("Cloud cover:", round(s2_item.properties.get("eo:cloud_cover"), 2))

print("\nUseful assets:")
for key in ["B02", "B03", "B04", "B08", "SCL", "visual"]:
    if key in s2_item.assets:
        print(key, "-", s2_item.assets[key].title)

#save the item 
s2_path = save_multiband_subset(
    item=s2_item,
    asset_keys=[
        "B02", #Blue
        "B03", #Green
        "B04", #Red
        "B08", #NIR
    ],
    bbox_wgs84=bbox_wgs84,
    output_path="../data/raw/imagery/sentinel2/estonia_field_0.tif",
)

[np.float64(26.48413253), np.float64(59.31079527), np.float64(26.52137214), np.float64(59.32976538)]
Found 12 Sentinel-2 item(s)
Selected item: S2B_MSIL2A_20210618T094029_R036_T35VMF_20210624T070731
Date: 2021-06-18 09:40:29.024000+00:00
Cloud cover: 0.06

Useful assets:
B02 - Band 2 - Blue - 10m
B03 - Band 3 - Green - 10m
B04 - Band 4 - Red - 10m
B08 - Band 8 - NIR - 10m
SCL - Scene classfication map (SCL)
visual - True color image
Saved: ../data/raw/imagery/sentinel2/estonia_field_0.tif (0.27 MB)


In [ ]:
m = leafmap.Map(height="600px")
m.add_basemap("Esri.WorldImagery") #This just regular satellite imagery, lets us see the crop fields well and check if they match up with the polygons

m.add_raster(
    str(s2_path),
    indexes=[3, 2, 1],
    layer_name="Sentinel-2 True Color"
)

m.add_raster(
    str(s2_path),
    indexes=[4, 3, 2],
    layer_name="Sentinel-2 False Color"
)

m.add_gdf(
    estonia_local,
    layer_name="Crop Fields"
)

m.add_layer_control()

m

Map(center=[59.320265500000005, 26.502783], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_i…

### Next thing to build is a split comparison between imagery and labels. 
This really achieves the same thing as before, just more fancy.

In [ ]:
m = leafmap.Map(height="600px")
m.split_map(
    left_layer="Esri.WorldImagery",
    right_layer=estonia_local,
    left_args={"indexes": [1, 2, 3]},
    left_label="Esri_Basemap",
    right_label="Crop Fields",
)
m

Map(center=[20, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text…

## The data doesn't have crop labels!

In [ ]:
estonia_gdf.head(5)
#it only has the parcel_id and the geometry, we need to combine with the other dataset, perform a left join with parcel_id as the common factor

,parcel_id,geometry
0,19994165,"POLYGON ((26.50243 59.31839, 26.50244 59.31843..."
1,19990783,"POLYGON ((24.54648 58.86884, 24.54674 58.86879..."
2,19990784,"POLYGON ((24.54597 58.86827, 24.54668 58.86816..."
3,19996106,"POLYGON ((27.42837 58.11975, 27.42839 58.11972..."
4,19990620,"POLYGON ((26.66816 57.82049, 26.66815 57.8205,..."


### Load the other dataset

In [ ]:
import pandas as pd

crop_labels = pd.read_parquet( #its a .parquet file
    "../data/raw/vectors/raw_data/labels/Estonia_labels.parquet"
)

print(crop_labels.shape)
print(crop_labels.columns)
print(crop_labels.head())

(176055, 3)
Index(['parcel_id', 'EC_hcat_c', 'EC_hcat_n'], dtype='object')
   parcel_id   EC_hcat_c                       EC_hcat_n
0   19994165  3302000000  pasture_meadow_grassland_grass
1   19990783  3302000000  pasture_meadow_grassland_grass
2   19990784  3302000000  pasture_meadow_grassland_grass
3   19996106  3301010101        winter_common_soft_wheat
4   19990620  3301090303                          clover


In [ ]:
print(estonia_gdf.columns)
print(estonia_gdf.head()) #find the common factor which is parcel_id

Index(['parcel_id', 'geometry'], dtype='object')
   parcel_id                                           geometry
0   19994165  POLYGON ((26.50243 59.31839, 26.50244 59.31843...
1   19990783  POLYGON ((24.54648 58.86884, 24.54674 58.86879...
2   19990784  POLYGON ((24.54597 58.86827, 24.54668 58.86816...
3   19996106  POLYGON ((27.42837 58.11975, 27.42839 58.11972...
4   19990620  POLYGON ((26.66816 57.82049, 26.66815 57.8205,...


In [ ]:
#perform a left join to merge the crop labels to their actual geometries
estonia_crops = estonia_gdf.merge(
    crop_labels,
    on="parcel_id",
    how="left"
)

In [ ]:
estonia_crops.head(5)
#now we have data that when we visualize it will tell us what kind of crop field it is

,parcel_id,geometry,EC_hcat_c,EC_hcat_n
0,19994165,"POLYGON ((26.50243 59.31839, 26.50244 59.31843...",3302000000,pasture_meadow_grassland_grass
1,19990783,"POLYGON ((24.54648 58.86884, 24.54674 58.86879...",3302000000,pasture_meadow_grassland_grass
2,19990784,"POLYGON ((24.54597 58.86827, 24.54668 58.86816...",3302000000,pasture_meadow_grassland_grass
3,19996106,"POLYGON ((27.42837 58.11975, 27.42839 58.11972...",3301010101,winter_common_soft_wheat
4,19990620,"POLYGON ((26.66816 57.82049, 26.66815 57.8205,...",3301090303,clover


### Lets visualize again this time with labeled crop data not just the parcel id. 

In [ ]:
#1. First make a subset again so we can visualize small
# Project to a metric CRS
estonia_projected = estonia_crops.to_crs("EPSG:32635")

center_field = estonia_projected.iloc[0]

distances = estonia_projected.geometry.centroid.distance(
    center_field.geometry.centroid
)

estonia_local_crops = estonia_crops.loc[distances[distances < 1000].index] 
estonia_local_crops.head(5)

,parcel_id,geometry,EC_hcat_c,EC_hcat_n
0,19994165,"POLYGON ((26.50243 59.31839, 26.50244 59.31843...",3302000000,pasture_meadow_grassland_grass
320,19994164,"POLYGON ((26.50425 59.31763, 26.50429 59.31763...",3302000000,pasture_meadow_grassland_grass
408,19994166,"POLYGON ((26.50002 59.31813, 26.50003 59.31813...",3302000000,pasture_meadow_grassland_grass
494,19994167,"POLYGON ((26.49936 59.31789, 26.49948 59.31779...",3302000000,pasture_meadow_grassland_grass
5335,20088271,"POLYGON ((26.50341 59.32013, 26.5034 59.31992,...",3301010401,winter_barley


In [ ]:
#2. Now that we have a subset of fields that are all next to one another we can get bounding box for the satellite
##Getting raster subset from bounding box
#Making bounding box 
bbox_wgs84 = list(estonia_local.total_bounds)

print(bbox_wgs84)

#Getting the new raster
search = pc_catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox_wgs84,
    datetime="2021-05-01/2021-09-30",
    query={"eo:cloud_cover": {"lt": 20}},
    max_items=20,
)

items = list(search.items())
print(f"Found {len(items)} Sentinel-2 item(s)")

#Choosing least cloudy scene
items_sorted = sorted(
    items,
    key=lambda item: item.properties.get("eo:cloud_cover", 999),
)

s2_item = pc.sign(items_sorted[0]) #this is now signed to STAC so I can access it 

print("Selected item:", s2_item.id)
print("Date:", s2_item.datetime)
print("Cloud cover:", round(s2_item.properties.get("eo:cloud_cover"), 2))

print("\nUseful assets:")
for key in ["B02", "B03", "B04", "B08", "SCL", "visual"]:
    if key in s2_item.assets:
        print(key, "-", s2_item.assets[key].title)

#save the item 
s2_path = save_multiband_subset(
    item=s2_item,
    asset_keys=[
        "B02", #Blue
        "B03", #Green
        "B04", #Red
        "B08", #NIR
    ],
    bbox_wgs84=bbox_wgs84,
    output_path="../data/raw/imagery/sentinel2/estonia_field_0.tif",
)

[np.float64(26.48413253), np.float64(59.31079527), np.float64(26.52137214), np.float64(59.32976538)]
Found 12 Sentinel-2 item(s)
Selected item: S2B_MSIL2A_20210618T094029_R036_T35VMF_20210624T070731
Date: 2021-06-18 09:40:29.024000+00:00
Cloud cover: 0.06

Useful assets:
B02 - Band 2 - Blue - 10m
B03 - Band 3 - Green - 10m
B04 - Band 4 - Red - 10m
B08 - Band 8 - NIR - 10m
SCL - Scene classfication map (SCL)
visual - True color image
Saved: ../data/raw/imagery/sentinel2/estonia_field_0.tif (0.27 MB)


In [ ]:
#3. Now we have the saved tif of a small bounding box of the local fields with the crop labels
#We can now map them. 
m = leafmap.Map(height="600px")
m.add_basemap("Esri.WorldImagery") #This just regular satellite imagery, lets us see the crop fields well and check if they match up with the polygons

m.add_raster(
    str(s2_path),
    indexes=[3, 2, 1],
    layer_name="Sentinel-2 True Color"
)

m.add_raster(
    str(s2_path),
    indexes=[4, 3, 2],
    layer_name="Sentinel-2 False Color"
)

m.add_gdf(
    estonia_local_crops,
    layer_name="Crop Fields"
)

m.add_layer_control()

m

Map(center=[59.320265500000005, 26.502783], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_i…

We can definitely see some differences visually in the fields. Most notably between winter wheat and grass pastures. However some fields which are labeled differently are very similar. At least to human eyes. 